# 05 - Eksperimen Peningkatan Model EduPath AI

Notebook ini digunakan untuk mencoba meningkatkan performa model Machine Learning dengan menambahkan feature baru dari dataset OULAD.

Model sebelumnya hanya menggunakan beberapa feature dasar seperti avg_score dan total_click. Pada eksperimen ini, akan ditambahkan feature lain seperti jumlah assessment, rata-rata waktu submit, jumlah hari aktif belajar, dan rata-rata klik per hari.

Tujuan eksperimen ini adalah melihat apakah feature tambahan dapat meningkatkan performa model.

## 1. Import Library

Tahap ini memanggil library yang dibutuhkan untuk membaca dataset, melakukan preprocessing, melatih model, dan mengevaluasi hasil model.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Load Dataset OULAD

Pada tahap ini kita membaca beberapa file utama dari dataset OULAD.

Dataset yang digunakan:
- studentInfo
- studentAssessment
- studentVle
- assessments
- vle

Dataset ini akan digunakan untuk membuat feature tambahan agar model memiliki informasi yang lebih lengkap.

In [2]:
student_info = pd.read_csv("../dataset/raw/studentInfo.csv")
student_assessment = pd.read_csv("../dataset/raw/studentAssessment.csv")
student_vle = pd.read_csv("../dataset/raw/studentVle.csv")
assessments = pd.read_csv("../dataset/raw/assessments.csv")
vle = pd.read_csv("../dataset/raw/vle.csv")

print("student_info:", student_info.shape)
print("student_assessment:", student_assessment.shape)
print("student_vle:", student_vle.shape)
print("assessments:", assessments.shape)
print("vle:", vle.shape)

student_info: (32593, 12)
student_assessment: (173912, 5)
student_vle: (10655280, 6)
assessments: (206, 6)
vle: (6364, 6)


## 3. Feature Engineering dari Student Assessment

Pada tahap ini membuat beberapa feature baru dari data assessment.

Feature yang dibuat:
- avg_score: rata-rata nilai siswa
- assessment_count: jumlah assessment yang dikerjakan siswa
- avg_date_submitted: rata-rata waktu submit assessment
- banked_count: jumlah assessment yang berstatus banked

Feature ini digunakan untuk melihat performa akademik dan pola pengerjaan assessment siswa.

In [3]:
assessment_features = (
    student_assessment
    .groupby("id_student")
    .agg(
        avg_score=("score", "mean"),
        assessment_count=("id_assessment", "count"),
        avg_date_submitted=("date_submitted", "mean"),
        banked_count=("is_banked", "sum")
    )
    .reset_index()
)

assessment_features.head()

,id_student,avg_score,assessment_count,avg_date_submitted,banked_count
0,6516,61.800000,5,111.600000,0
1,8462,87.000000,7,23.000000,4
2,11391,82.000000,5,112.400000,0
3,23629,82.500000,4,55.750000,0
4,23698,74.444444,9,133.444444,0


## 4. Feature Engineering dari Student VLE

Pada tahap ini membuat feature dari aktivitas siswa di Virtual Learning Environment.

Feature yang dibuat:
- total_click: total klik siswa selama belajar
- active_days: jumlah hari siswa aktif
- avg_click_per_day: rata-rata klik per hari aktif
- max_click_day: jumlah klik tertinggi dalam satu hari

Feature ini digunakan untuk melihat keterlibatan siswa dalam proses pembelajaran.

In [4]:
vle_features = (
    student_vle
    .groupby("id_student")
    .agg(
        total_click=("sum_click", "sum"),
        active_days=("date", "nunique"),
        avg_click_per_day=("sum_click", "mean"),
        max_click_day=("sum_click", "max")
    )
    .reset_index()
)

vle_features.head()

,id_student,total_click,active_days,avg_click_per_day,max_click_day
0,6516,2791,159,4.216012,49
1,8462,656,56,2.157895,16
2,11391,934,40,4.765306,76
3,23629,161,16,2.728814,13
4,23698,910,70,2.983607,78


## 5. Pemilihan Data Utama Siswa

Pada tahap ini mengambil kolom penting dari studentInfo.

Kolom final_result digunakan untuk membuat target needs_remedial, tetapi tidak akan digunakan sebagai feature model karena dapat menyebabkan data leakage.

In [5]:
student_master = student_info[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "gender",
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability",
        "final_result"
    ]
]

student_master.head()

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


## 6. Menggabungkan Dataset

tahap ini data siswa, feature assessment, dan feature aktivitas belajar digabungkan berdasarkan id_student.

Hasil penggabungan ini akan menjadi dataset eksperimen untuk mencoba meningkatkan performa model.

In [6]:
improved_df = student_master.merge(
    assessment_features,
    on="id_student",
    how="inner"
)

improved_df = improved_df.merge(
    vle_features,
    on="id_student",
    how="inner"
)

improved_df.head()

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avg_score,assessment_count,avg_date_submitted,banked_count,total_click,active_days,avg_click_per_day,max_click_day
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,5,112.4,0,934,40,4.765306,76
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,5,114.2,0,1435,80,3.337209,23
2,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,5,112.2,0,2158,123,3.254902,22
3,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,5,125.6,0,1034,70,2.937500,22
4,38053,AAA,2013J,M,Wales,A Level or Equivalent,80-90%,35-55,0,60,N,Pass,68.0,5,116.2,0,2445,143,3.381743,22


In [7]:
print("Shape improved_df:", improved_df.shape)

Shape improved_df: (26721, 20)


## 7. Membuat Target needs_remedial

Target needs_remedial dibuat dari kolom final_result.

Aturan:
- Fail = 1
- Withdrawn = 1
- Pass = 0
- Distinction = 0

Kolom final_result tidak digunakan sebagai feature karena target dibuat dari kolom tersebut.

In [8]:
improved_df["needs_remedial"] = improved_df["final_result"].apply(
    lambda x: 1 if x in ["Fail", "Withdrawn"] else 0
)

print(improved_df["needs_remedial"].value_counts())

needs_remedial
0    15381
1    11340
Name: count, dtype: int64


## 8. Cleaning Data

Bagian ini digunakan untuk mengecek data kosong dan membersihkan data sebelum masuk ke model.

Data kosong perlu ditangani agar proses training tidak error.

In [9]:

print("Missing values sebelum cleaning:")
print(improved_df.isnull().sum())

improved_df = improved_df.dropna()

print("\nMissing values setelah cleaning:")
print(improved_df.isnull().sum())

print("\nShape setelah cleaning:")
print(improved_df.shape)

Missing values sebelum cleaning:
id_student                 0
code_module                0
code_presentation          0
gender                     0
region                     0
highest_education          0
imd_band                1012
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
avg_score                 19
assessment_count           0
avg_date_submitted         0
banked_count               0
total_click                0
active_days                0
avg_click_per_day          0
max_click_day              0
needs_remedial             0
dtype: int64

Missing values setelah cleaning:
id_student              0
code_module             0
code_presentation       0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0

## 9. Encoding Data Kategori

Beberapa kolom masih berbentuk teks, seperti gender, region, education, dan disability.

Model Machine Learning hanya bisa membaca angka, jadi kolom kategori perlu diubah menjadi angka.

In [10]:
categorical_columns = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

encoder = LabelEncoder()

for col in categorical_columns:
    improved_df[col] = encoder.fit_transform(improved_df[col])

improved_df.head()

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,final_result,avg_score,assessment_count,avg_date_submitted,banked_count,total_click,active_days,avg_click_per_day,max_click_day,needs_remedial
0,11391,0,1,1,0,1,9,2,0,240,...,Pass,82.0,5,112.4,0,934,40,4.765306,76,0
1,28400,0,1,0,6,1,2,1,0,60,...,Pass,66.4,5,114.2,0,1435,80,3.337209,23,0
2,31604,0,1,0,7,0,5,1,0,60,...,Pass,76.0,5,112.2,0,2158,123,3.254902,22,0
3,32885,0,1,0,11,2,5,0,0,60,...,Pass,54.4,5,125.6,0,1034,70,2.937500,22,0
4,38053,0,1,1,10,0,8,1,0,60,...,Pass,68.0,5,116.2,0,2445,143,3.381743,22,0


## 10. Menentukan Feature dan Target

Feature adalah kolom yang digunakan model untuk belajar.

Target adalah kolom yang ingin diprediksi.

Pada eksperimen ini, target yang digunakan adalah needs_remedial.

In [11]:
features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
    "avg_score",
    "assessment_count",
    "avg_date_submitted",
    "banked_count",
    "total_click",
    "active_days",
    "avg_click_per_day",
    "max_click_day"
]

target = "needs_remedial"

X = improved_df[features]
y = improved_df[target]

print("Feature X:")
display(X.head())

print("\nTarget y:")
display(y.head())

print("\nShape X:", X.shape)
print("Shape y:", y.shape)

Feature X:


,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,avg_score,assessment_count,avg_date_submitted,banked_count,total_click,active_days,avg_click_per_day,max_click_day
0,0,1,1,0,1,9,2,0,240,0,82.0,5,112.4,0,934,40,4.765306,76
1,0,1,0,6,1,2,1,0,60,0,66.4,5,114.2,0,1435,80,3.337209,23
2,0,1,0,7,0,5,1,0,60,0,76.0,5,112.2,0,2158,123,3.254902,22
3,0,1,0,11,2,5,0,0,60,0,54.4,5,125.6,0,1034,70,2.937500,22
4,0,1,1,10,0,8,1,0,60,0,68.0,5,116.2,0,2445,143,3.381743,22



Target y:


0    0
1    0
2    0
3    0
4    0
Name: needs_remedial, dtype: int64


Shape X: (25690, 18)
Shape y: (25690,)


## 11. Split Data

Data dibagi menjadi data training dan data testing.

Data training dipakai untuk melatih model.

Data testing dipakai untuk mengecek kemampuan model pada data yang belum pernah dilihat.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (20552, 18)
X_test : (5138, 18)
y_train: (20552,)
y_test : (5138,)


## 12. Training Random Forest Improved

Model pertama yang dicoba adalah Random Forest.

Model ini digunakan karena cukup kuat untuk data tabular dan bisa membaca hubungan antar feature dengan lebih baik.

In [13]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_cm = confusion_matrix(y_test, rf_pred)

print("Random Forest Improved")
print("Accuracy:", round(rf_accuracy * 100, 2), "%")

print("\nConfusion Matrix:")
print(rf_cm)

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Improved
Accuracy: 89.92 %

Confusion Matrix:
[[2783  147]
 [ 371 1837]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      2930
           1       0.93      0.83      0.88      2208

    accuracy                           0.90      5138
   macro avg       0.90      0.89      0.90      5138
weighted avg       0.90      0.90      0.90      5138



## 13. Training Gradient Boosting

Model kedua yang dicoba adalah Gradient Boosting.

Model ini dipakai sebagai pembanding karena sering cukup baik untuk data tabular.

In [14]:
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

gb_accuracy = accuracy_score(y_test, gb_pred)
gb_cm = confusion_matrix(y_test, gb_pred)

print("Gradient Boosting")
print("Accuracy:", round(gb_accuracy * 100, 2), "%")

print("\nConfusion Matrix:")
print(gb_cm)

print("\nClassification Report:")
print(classification_report(y_test, gb_pred))

Gradient Boosting
Accuracy: 89.61 %

Confusion Matrix:
[[2796  134]
 [ 400 1808]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      2930
           1       0.93      0.82      0.87      2208

    accuracy                           0.90      5138
   macro avg       0.90      0.89      0.89      5138
weighted avg       0.90      0.90      0.89      5138



## 14. Perbandingan Hasil Model

Bagian ini digunakan untuk membandingkan hasil model lama dan model eksperimen.

Yang dilihat bukan hanya accuracy, tetapi juga recall pada kelas 1 karena kelas 1 berarti pengguna perlu remedial.

In [15]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression Baseline",
        "Random Forest Baseline",
        "Random Forest Improved",
        "Gradient Boosting"
    ],
    "Accuracy": [
        73.92,
        73.13,
        round(rf_accuracy * 100, 2),
        round(gb_accuracy * 100, 2)
    ]
})

model_comparison

,Model,Accuracy
0,Logistic Regression Baseline,73.92
1,Random Forest Baseline,73.13
2,Random Forest Improved,89.92
3,Gradient Boosting,89.61


### Interpretasi Hasil Eksperimen

Setelah menambahkan feature baru dari data assessment dan aktivitas VLE, performa model meningkat cukup besar dibandingkan model baseline.

Random Forest Improved memperoleh accuracy sebesar 89.92%, sedangkan Gradient Boosting memperoleh accuracy sebesar 89.61%.

Peningkatan ini menunjukkan bahwa feature tambahan seperti jumlah assessment, rata-rata waktu submit, jumlah hari aktif belajar, dan pola klik belajar membantu model membaca kondisi pengguna dengan lebih baik.

Untuk tahap ini, Random Forest Improved menjadi model terbaik karena memiliki accuracy paling tinggi dibandingkan model lainnya.

Namun, hasil ini tetap perlu dilihat bersama confusion matrix dan classification report, terutama recall pada kelas 1 karena kelas tersebut menunjukkan pengguna yang perlu remedial.

## 15. Feature Importance Model Improved

Bagian ini digunakan untuk melihat feature mana yang paling berpengaruh terhadap prediksi model.

Feature importance membantu kita memahami alasan model bisa meningkat setelah ditambahkan feature baru.

In [16]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

,Feature,Importance
12,avg_date_submitted,0.328675
11,assessment_count,0.151646
10,avg_score,0.121128
15,active_days,0.083562
14,total_click,0.058813
0,code_module,0.045450
17,max_click_day,0.041022
16,avg_click_per_day,0.040221
3,region,0.021561
5,imd_band,0.020715


## 16. Menyimpan Model Improved

Model Random Forest Improved disimpan agar dapat digunakan kembali pada tahap berikutnya.

Model ini menjadi model eksperimen terbaik pada dataset OULAD setelah penambahan feature engineering.

In [18]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "edupath_oulad_improved_rf_model.pkl"

joblib.dump(rf_model, model_path)

print("Model improved berhasil disimpan ke:")
print(model_path)

Model improved berhasil disimpan ke:
d:\Kuliah\Dicoding\Github\EduPath-AI\models\edupath_oulad_improved_rf_model.pkl


### Interpretasi Feature Importance

Berdasarkan hasil feature importance, fitur yang paling berpengaruh pada model improved adalah `avg_date_submitted`, `assessment_count`, dan `avg_score`.

Hal ini menunjukkan bahwa pola pengerjaan assessment memiliki pengaruh besar terhadap prediksi kebutuhan remedial. Siswa yang memiliki pola submit tertentu, jumlah assessment tertentu, dan rata-rata nilai tertentu lebih mudah dikenali oleh model.

Fitur aktivitas belajar seperti `active_days`, `total_click`, `max_click_day`, dan `avg_click_per_day` juga berpengaruh, meskipun tidak sebesar fitur assessment.

Dari hasil ini dapat disimpulkan bahwa peningkatan performa model terjadi karena model tidak hanya melihat nilai rata-rata dan total klik, tetapi juga melihat pola pengerjaan assessment dan aktivitas belajar.

Model Random Forest Improved berhasil meningkatkan accuracy menjadi 89.92% setelah ditambahkan feature baru dari assessment dan aktivitas VLE. Namun, model ini masih perlu dianalisis lebih lanjut dari sisi waktu penggunaan fitur, karena beberapa fitur seperti rata-rata waktu submit dan jumlah assessment lebih cocok digunakan setelah pengguna memiliki riwayat belajar yang cukup.

In [19]:
print("Random Forest Improved")
print("Accuracy:", round(rf_accuracy * 100, 2), "%")

print("\nConfusion Matrix:")
print(rf_cm)

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Improved
Accuracy: 89.92 %

Confusion Matrix:
[[2783  147]
 [ 371 1837]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      2930
           1       0.93      0.83      0.88      2208

    accuracy                           0.90      5138
   macro avg       0.90      0.89      0.90      5138
weighted avg       0.90      0.90      0.90      5138



### Interpretasi Evaluasi Random Forest Improved

Model Random Forest Improved memperoleh accuracy sebesar 89.92%.

Hasil ini lebih tinggi dibandingkan model baseline sebelumnya yang berada di sekitar 73%. Peningkatan ini terjadi setelah dilakukan penambahan feature baru dari data assessment dan aktivitas VLE.

Pada confusion matrix, model berhasil memprediksi 1837 data pengguna yang perlu remedial dengan benar. Namun masih terdapat 371 data pengguna yang sebenarnya perlu remedial tetapi diprediksi tidak perlu remedial.

Recall pada kelas 1 sebesar 83%. Artinya model mampu menangkap sebagian besar pengguna yang membutuhkan remedial, tetapi masih ada beberapa pengguna yang belum terdeteksi.

Untuk tahap eksperimen, hasil ini sudah cukup baik karena menunjukkan bahwa feature engineering dapat meningkatkan performa model secara signifikan.